# Part 2: The Three Categories of Machine Learning
**⏱ This section takes approximately 30 minutes.**

---

## Scenario: Sarah gets more projects

Sarah's manager at **NorthStar Retail** is impressed by the review-classification work. On Monday, three more problems land on her desk:

> **Problem A:** *"We have 200,000 customers. Group them into 4 or 5 'types' so marketing can target them differently."* (No one has defined the types in advance.)
>
> **Problem B:** *"Predict which of our current customers will stop shopping with us in the next 3 months."* (We have 2 years of labelled customer history — who churned and who stayed.)
>
> **Problem C:** *"Build a bot that plays through our returns-process flow and finds the fastest path for a refund."* (We'll let it try and we'll reward the fast paths.)

Every ML problem in the world falls into one of three categories. Each of Sarah's three problems is in a *different* category.

**By the end of this section you will:**
- Recognise the three categories of ML and what each needs
- Match a business problem to the right category
- Understand why supervised learning is the most common in business

> *Sarah Chen · Customer Experience Analyst · NorthStar Retail · January 2023 (still Monday).*
> You just saw ML work on Sarah's reviews. But ML is a family of methods, not one thing. This notebook shows you the three main categories — and which one Sarah is actually using.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs, make_classification
import warnings

# scikit-learn prints internal deprecation notes that are not errors.
# We silence them here so the output stays clean for learning.
warnings.filterwarnings("ignore")

np.random.seed(42)
print("✅ Ready!")

## 🎯 Category 1 — Supervised Learning

**The idea in plain English:**
> You have inputs *and* known correct answers. The model learns to map inputs to answers so it can predict answers for new, unseen inputs.

**Real-world analogy:** A student studying past exam papers *with answer keys*. They see many questions and the correct answers, and over time learn to answer new questions on their own.

**Why it matters for ML:** Supervised learning is the workhorse of business ML. Churn prediction, fraud detection, demand forecasting, sentiment analysis — all supervised. **It's the category we'll spend the most time on in this course.**

### Demo: a tiny supervised classifier

Sarah's **Problem B** was: *predict which customers will churn.* Let's simulate it with a tiny dataset.

In [ ]:
# Fake customer data: 2 features (monthly spend, months since last visit), 1 label (churned or not)
X, y = make_classification(
    n_samples=200, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=1.5, random_state=42,
)

df = pd.DataFrame(X, columns=["monthly_spend", "months_since_visit"])
df["churned"] = y

print("Sample of our training data (feature columns + label):")
df.head()

In [ ]:
# Train a supervised classifier
model = LogisticRegression().fit(X, y)

# Now predict for two new customers we've never seen before
new_customers = pd.DataFrame({
    "monthly_spend": [120, 35],
    "months_since_visit": [0.5, 6],
})
predictions = model.predict(new_customers)
probabilities = model.predict_proba(new_customers)[:, 1]

for i, (_, c) in enumerate(new_customers.iterrows()):
    p = "will churn" if predictions[i] else "will stay"
    print(f"Customer with spend=${c['monthly_spend']}, months_since_visit={c['months_since_visit']}:")
    print(f"  Prediction: {p}  (churn probability: {probabilities[i]:.0%})\n")

### 💡 What do you notice?

- **The model saw 200 labelled examples and learned a boundary** between "churner" and "non-churner" in feature-space.
- For every new prediction it gives a **probability** — not just a yes/no. Same mechanism as the review confidence scores in Part 1.
- **We needed labelled historical data** to make this work. If Sarah didn't know who churned in the past, this category wouldn't apply.

**Back to our scenario:** Problem B fits supervised learning perfectly because NorthStar already knows who has and hasn't churned over the last 2 years.

## 🎯 Category 2 — Unsupervised Learning

**The idea in plain English:**
> You have inputs *but no correct answers*. The model finds structure — groups, patterns, anomalies — in the data on its own.

**Real-world analogy:** A librarian faced with a crate of new books, none labelled by genre. She groups them by *"these feel similar"* — romance here, mystery there — even though no one told her what the genres are.

**Why it matters for ML:** Unsupervised learning is for **exploration**: customer segmentation, anomaly detection, document clustering. **Covered in L06.**

### Demo: a tiny clustering

Sarah's **Problem A** was: *group 200,000 customers into 4–5 types.* Let's simulate it.

In [ ]:
# Fake customer data — no labels!
X_customers, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

# Ask the algorithm to find 4 groups
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
groups = kmeans.fit_predict(X_customers)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(X_customers[:, 0], X_customers[:, 1], c="gray", s=20)
axes[0].set_title("Before: 300 customers, no labels")

axes[1].scatter(X_customers[:, 0], X_customers[:, 1], c=groups, cmap="viridis", s=20)
axes[1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], c="red", marker="X", s=150, label="group centers")
axes[1].set_title("After: model found 4 natural groups")
axes[1].legend()
plt.tight_layout()
plt.show()

### 💡 What do you notice?

- **We never told the algorithm which customers belonged to which group.** It found the groups itself based on the two features.
- The output is *"customer belongs to group 0 / 1 / 2 / 3"* — but the **groups don't have names** yet. Sarah's next step is to inspect each group and figure out what makes them similar.
- Unsupervised is harder to evaluate: *did we get the "right" 4 groups?* There's no answer key.

**Back to our scenario:** Problem A (group 200,000 customers into types) fits here — NorthStar does **not** have predefined "customer types." They want the data to reveal them.

## 🎯 Category 3 — Reinforcement Learning

**The idea in plain English:**
> An agent *takes actions in an environment*, gets *rewards or penalties*, and adjusts its behaviour over time to maximise reward.

**Real-world analogy:** Training a dog with treats. Sit → treat. Bark at the mailman → no treat. Over time the dog learns which actions pay off.

**Why it matters for ML:** Reinforcement learning is behind game-playing AIs, robotic control, recommendation systems at scale, and parts of how ChatGPT was trained. **We don't build RL systems in this course** — it's a big field — but you need to recognise the category when you see it.

Sarah's **Problem C** (bot that explores the returns-process and gets rewarded for fast paths) is a reinforcement-learning problem.

### When would you use RL?

Use RL when:
- You **can't list correct answers** in advance (what is the *right* move at chess move #12?)
- You can only **give feedback on what the system does** (+1 for win, 0 for draw, -1 for loss)
- The system needs to **explore** to discover good behaviour

Don't use RL when:
- You already have labelled data showing the right answer (use supervised learning — much cheaper)
- You just want to understand what's in the data (use unsupervised)

## 🧩 Comparison at a glance

In [ ]:
categories = pd.DataFrame({
    "Supervised":   ["Features + Labels",  "Predict labels",         "Churn, fraud, sentiment", "L03 L04 L05 L08 L09"],
    "Unsupervised": ["Features only",       "Find structure",        "Segmentation, anomaly",    "L06"],
    "Reinforcement":["Environment + reward","Maximize reward",        "Games, robotics, recsys",  "—"],
}, index=["Data you need", "Goal", "Business example", "Covered in"]).T

categories

## 🧠 Apply it — match the problem to the category

For each problem below, pick the category that fits best:

1. Detect credit-card fraud — you have millions of past transactions labelled fraud / not-fraud.
2. Group 500k email subscribers into 3–5 "interest segments" — no segments defined yet.
3. Train a warehouse robot to move packages — reward for completed moves, penalty for drops.
4. Predict monthly sales for a new store location — you have historical monthly sales for 50 similar stores.
5. Find unusual patterns in server logs that might indicate a security breach — no labels for "breach."

*Write your answers in the cell below.*

### ⏸️ Pause and Predict

Write your 5 answers, then read the sample answer below:

1.
2.
3.
4.
5.

### Sample answer

1. **Supervised.** Labelled history → predict labels for new transactions.
2. **Unsupervised.** No labels → discover groups.
3. **Reinforcement.** Reward signal + trial-and-error behaviour.
4. **Supervised.** Historical monthly sales are labels (the model learns to predict future sales from store features).
5. **Unsupervised** (specifically anomaly detection — a specialisation we'll touch on in L06). No labelled "breach" examples exist, but we can flag things that look unlike everything else.

## 💼 Which category is most common in business — and why?

In business ML projects, the split is roughly:

| Category | Share of business ML projects (rough estimate) |
|---|---|
| Supervised | ~70–80% |
| Unsupervised | ~15–25% |
| Reinforcement | ~1–5% |

**Why supervised dominates:**
- Most business data arrives *with labels* (did the customer churn? did the transaction get reported as fraud? did the user click?).
- Evaluation is straightforward — you can measure accuracy on held-out data.
- The techniques are mature and reliable.

**Why RL is rare in business:**
- Needs an environment the agent can safely interact with — usually a simulator, which is expensive to build.
- Training takes millions of trials.
- Most business decisions can be reframed as supervised if you're willing to collect labels.

**This is why this course spends 8 of 10 lessons on supervised techniques or their deep-learning extensions.**

## Reflection questions

Take 3 minutes. Answer in the cell below.

1. For Sarah's **review-classification** task (from Part 1), which category does it fit, and why?
2. Pick a business problem from your own work or life. Which category would it fit, and what would you need to collect before you could train a model?
3. Under what conditions would you *choose* unsupervised learning over supervised, even if you had some labels?

*Your answers:*

1.
2.
3.

## ✅ Section 2 Summary

| Category | Needs labels? | Goal | When to use |
|---|---|---|---|
| **Supervised** | Yes | Predict labels for new data | Most business ML — churn, fraud, forecasting, sentiment |
| **Unsupervised** | No | Discover structure | Segmentation, anomaly detection, exploratory work |
| **Reinforcement** | No (but needs reward) | Maximise long-term reward | Games, robotics, large-scale recommendations |

**Key insight for our scenario:**
> Sarah's three new projects span all three categories — but the one her company will actually deploy first is the supervised churn model, because the data already exists and the outcome is directly actionable.

---
**Up next → Section 3:** The full ML workflow — how a project actually runs from business question to production model.
Open `04_ml_workflow.ipynb`